# Clase 6: Secuencias y redes neuronales recurrentes

## Objetivos de aprendizaje

Al terminar esta clase, tú:

- Explicas por qué un perceptrón multicapa (MLP) no capta bien datos secuenciales
- Implementas a mano el paso recurrente de una red neuronal recurrente (RNN) simple, con un bucle en PyTorch
- Identificas el problema del gradiente que desaparece en secuencias largas
- Entrenas una red LSTM (memoria a corto-largo plazo, del inglés *Long Short-Term Memory*) sobre una serie temporal sintética
- Comparas la predicción del modelo con los valores reales de la serie, de forma gráfica

## Datos que dependen del orden

Un perceptrón multicapa (MLP) espera un vector de tamaño fijo. Si lo entrenas con frases de cinco palabras, no sabe qué hacer con una frase de ocho palabras.

Además, un MLP normalmente concatena o suma sus entradas sin distinguir posiciones. Por eso no distingue "el gato persigue al perro" de "el perro persigue al gato": mismas palabras, significado opuesto, orden distinto.

Necesitas una arquitectura que:

- Acepte secuencias de longitud variable
- Respete el orden de los elementos
- Recuerde información de pasos anteriores mientras procesa el paso actual

Esa arquitectura es la red neuronal recurrente (RNN).

## El estado oculto de una RNN

Piensa en cómo lees esta misma frase, palabra por palabra. No interpretas cada palabra de forma aislada: mantienes en la cabeza un resumen mental de lo que leíste hasta ese punto, y ese resumen condiciona cómo interpretas la palabra nueva.

Una RNN hace algo parecido. Mantiene un vector de estado oculto, hₜ, que actualiza en cada paso temporal combinando la entrada actual con el estado del paso anterior:

**hₜ = tanh(Wₕ · hₜ₋₁ + Wₓ · xₜ + bₕ)**

Wₕ y Wₓ son matrices de pesos que se comparten en todos los pasos temporales: la misma regla de actualización se aplica en el paso tres que en el paso 300. Eso permite procesar secuencias de cualquier longitud con un número fijo de parámetros.

Implementa este paso a mano, con un bucle en PyTorch, para ver cómo evoluciona el estado oculto conforme entra cada nuevo elemento de la secuencia.

In [ ]:
import torch

torch.manual_seed(42)

tam_entrada = 4   # dimension de cada elemento de la secuencia
tam_oculto = 3    # dimension del estado oculto

# Pesos aleatorios, como los tendria una capa RNN recien creada, sin entrenar
Wx = torch.randn(tam_oculto, tam_entrada) * 0.5
Wh = torch.randn(tam_oculto, tam_oculto) * 0.5
bh = torch.zeros(tam_oculto)

# Una secuencia de ejemplo: seis pasos temporales, cuatro valores por paso
secuencia = torch.randn(6, tam_entrada)

h = torch.zeros(tam_oculto)  # estado oculto inicial h_0, en cero

print("Evolucion del estado oculto paso a paso:\n")
for t, x_t in enumerate(secuencia):
    # el mismo calculo, con los mismos pesos Wx y Wh, se repite en cada paso temporal
    h = torch.tanh(Wh @ h + Wx @ x_t + bh)
    print(f"Paso {t + 1}: h = {h.numpy().round(3)}")

## El gradiente que desaparece

Simplifica la recurrencia al máximo: ignora entradas y sesgos, y supón que Wₕ se reduce a un único número w. La actualización hₜ = w · hₜ₋₁ se desenrolla como:

**hₜ = wᵗ · h₀**

Si el valor absoluto de w supera uno, wᵗ crece sin control conforme avanza t: es el gradiente explosivo. Si el valor absoluto de w es menor que uno, wᵗ decae hacia cero: es el gradiente que se desvanece.

En la práctica, la actualización pasa además por una tangente hiperbólica, cuya derivada nunca supera uno. El gradiente que se retropropaga a través de muchos pasos temporales es, en esencia, un producto de muchos factores como este. Basta con que cada factor sea algo menor que uno para que, tras decenas o cientos de pasos, el producto se aproxime a cero.

La consecuencia es grave: si el gradiente que llega a un paso lejano es prácticamente nulo, esa parte de la red deja de recibir señal de aprendizaje, y el modelo no aprende dependencias distantes, aunque tenga capacidad teórica para representarlas.

## LSTM y GRU al rescate

La red LSTM (memoria a corto-largo plazo) separa dos flujos de información: un estado de celda, cₜ, que actúa como memoria a largo plazo con actualizaciones mayormente aditivas, y un estado oculto hₜ, que se calcula a partir del estado de celda en cada paso.

Tres puertas, cada una con activación sigmoide, regulan el acceso a esa memoria:

- **Puerta de olvido**: decide qué parte del estado de celda anterior conservas
- **Puerta de entrada**: decide qué información nueva incorporas
- **Puerta de salida**: decide qué parte del estado de celda expones como estado oculto

La actualización del estado de celda, cₜ = fₜ ⊙ cₜ₋₁ + iₜ ⊙ c̃ₜ, es en gran parte una suma, no una reescritura completa. Ese carácter aditivo es la clave: cuando la puerta de olvido está cerca de uno, el gradiente fluye hacia atrás casi sin atenuarse.

La GRU (unidad recurrente con puertas, del inglés *Gated Recurrent Unit*) simplifica la LSTM: fusiona el estado de celda y el oculto en un único vector, y reduce las puertas a dos (actualización y reinicio). Tiene menos parámetros, entrena algo más rápido y, en muchas tareas, rinde de forma comparable.

Entrena ahora una LSTM pequeña con `torch.nn.LSTM` sobre una serie temporal sintética: un seno con ruido. Como función de pérdida usas el error cuadrático medio (MSE, del inglés *Mean Squared Error*), habitual en tareas de regresión.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# Serie temporal sintetica: un seno con ruido gaussiano
pasos_totales = 500
t = np.linspace(0, 50, pasos_totales)
serie = (np.sin(t) + np.random.normal(0, 0.1, size=pasos_totales)).astype(np.float32)

plt.figure(figsize=(10, 3))
plt.plot(serie)
plt.title("Serie temporal sintetica: seno con ruido")
plt.xlabel("Paso temporal")
plt.ylabel("Valor")
plt.show()

longitud_ventana = 20  # cuantos pasos pasados usa el modelo para predecir el siguiente

def crear_ventanas(serie, longitud_ventana):
    X, y = [], []
    for i in range(len(serie) - longitud_ventana):
        X.append(serie[i:i + longitud_ventana])
        y.append(serie[i + longitud_ventana])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = crear_ventanas(serie, longitud_ventana)

# Reservamos una parte para entrenar y el resto para evaluar, respetando el orden temporal
corte = int(len(X) * 0.8)
X_entreno, X_prueba = X[:corte], X[corte:]
y_entreno, y_prueba = y[:corte], y[corte:]

# nn.LSTM espera tensores con forma (lote, tiempo, caracteristicas)
X_entreno_t = torch.tensor(X_entreno).unsqueeze(-1)
y_entreno_t = torch.tensor(y_entreno).unsqueeze(-1)
X_prueba_t = torch.tensor(X_prueba).unsqueeze(-1)
y_prueba_t = torch.tensor(y_prueba).unsqueeze(-1)

print("Forma de X_entreno:", X_entreno_t.shape)

In [ ]:
class ModeloLSTM(nn.Module):
    def __init__(self, tam_oculto=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=tam_oculto, batch_first=True)
        self.salida = nn.Linear(tam_oculto, 1)

    def forward(self, x):
        salida_lstm, _ = self.lstm(x)
        ultimo_paso = salida_lstm[:, -1, :]  # el ultimo estado oculto resume toda la ventana
        return self.salida(ultimo_paso)

modelo = ModeloLSTM()
perdida_fn = nn.MSELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)

num_epocas = 150
historial_perdida = []

for epoca in range(num_epocas):
    modelo.train()
    prediccion = modelo(X_entreno_t)
    perdida = perdida_fn(prediccion, y_entreno_t)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
    historial_perdida.append(perdida.item())
    if (epoca + 1) % 30 == 0:
        print(f"Epoca {epoca + 1}/{num_epocas} - perdida: {perdida.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(historial_perdida)
plt.xlabel("Epoca")
plt.ylabel("Perdida (MSE)")
plt.title("Curva de perdida durante el entrenamiento")
plt.show()

In [ ]:
modelo.eval()
with torch.no_grad():
    prediccion_prueba = modelo(X_prueba_t).squeeze().numpy()

plt.figure(figsize=(10, 4))
plt.plot(y_prueba, label="Valor real")
plt.plot(prediccion_prueba, label="Prediccion de la LSTM")
plt.xlabel("Paso temporal (conjunto de prueba)")
plt.ylabel("Valor")
plt.title("Prediccion frente a valores reales")
plt.legend()
plt.show()

## Resumen de puntos clave

- Una secuencia es una colección ordenada de elementos de longitud variable: el texto, el audio y las series temporales son ejemplos habituales
- Un MLP exige entradas de tamaño fijo y no captura el orden; una RNN mantiene un estado oculto que resume lo visto hasta el momento
- El estado oculto se actualiza con hₜ = tanh(Wₕ · hₜ₋₁ + Wₓ · xₜ + bₕ), con los mismos pesos compartidos en todos los pasos temporales
- En secuencias largas, la RNN simple sufre el problema del gradiente que desaparece o explota, y deja de aprender dependencias lejanas
- La LSTM introduce un estado de celda con actualización aditiva y tres puertas (olvido, entrada, salida) que protegen la memoria a largo plazo
- La GRU simplifica la LSTM con dos puertas (actualización y reinicio) y menos parámetros
- Ambas arquitecturas mitigan el desvanecimiento del gradiente porque dejan fluir la información con cambios mínimos a través de muchos pasos

**Abre ahora el notebook de práctica, `../notebooks_practica/clase06_practica_secuencias_redes_recurrentes.ipynb`, y pon en marcha lo que acabas de ver.**